# UK Wind Generation — Forecast Error & Reliability Analysis

## Background

Elexon is part of the UK electricity market infrastructure. They run the Balancing and Settlement Code (BSC) system, which is responsible for reconciling how much electricity each generator actually produced versus what was contracted, and settling the financial difference. The data they publish via the BMRS API reflects real grid activity.

This notebook uses two of their public endpoints (no API key required):

| Endpoint | What it contains | Fields used |
|----------|-----------------|-------------|
| `FUELHH` | Half-hourly actual generation per fuel type | `startTime`, `generation`, `fuelType` (filtered to `WIND`) |
| `WINDFOR` | Wind generation forecasts with publish timestamps | `startTime`, `publishTime`, `generation` |

**`startTime`** is the beginning of the 30-minute settlement window the reading covers — this is what gets plotted on the x-axis.

**`publishTime`** (forecasts only) is when the forecast was created. The difference between the two is the **forecast horizon**:

```
horizon = startTime - publishTime
```

A forecast for 18:00 published at 14:00 has a 4-hour horizon. The dashboard exposes this as a slider (0–48 hours, default 4 hours). For each target time, it shows the latest forecast whose horizon is at least the selected value.

Data is restricted to **January 2025 onwards** per the project specification, with forecast horizons bounded to **0–48 hours**.

> **Note:** There are parts of the Elexon data model that aren't fully clear from the documentation alone. The implementation is based on observed API behaviour and may not be correct in all edge cases.

---

## Objectives

1. **Forecast error analysis** — characterise how accurate the wind forecasts are, and how accuracy varies with horizon and time of day
2. **Reliability analysis** — determine how many MW of wind generation can be dependably expected, with a reasoned recommendation

## 0. Setup & Imports

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timezone

# Consistent plot styling throughout the notebook
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 120

## 1. Data Fetching

The two endpoints used:

```
GET https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH/stream
    ?publishDateTimeFrom=2025-01-01T00:00:00Z
    &publishDateTimeTo=2025-03-31T23:30:00Z

GET https://data.elexon.co.uk/bmrs/api/v1/datasets/WINDFOR/stream
    ?publishDateTimeFrom=2025-01-01T00:00:00Z
    &publishDateTimeTo=2025-03-31T23:30:00Z
```

The `publishDateTimeFrom` / `publishDateTimeTo` parameters filter by when records were published to the feed, not the settlement time they cover.

The fetch functions below handle the HTTP call and basic field extraction only — no analysis logic lives here.

**Date range chosen:** January–March 2025. Three months gives enough data for statistically stable percentiles and captures variation within a single quarter. Extending the window is straightforward by changing `DATE_FROM` / `DATE_TO` below.

In [ ]:
BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1"

def fetch_actuals(date_from: str, date_to: str) -> pd.DataFrame:
    """
    Fetches half-hourly actual wind generation from the FUELHH endpoint.
    Returns a DataFrame with columns: startTime, actual_mw
    """
    params = {
        "publishDateTimeFrom": date_from,
        "publishDateTimeTo": date_to,
        "format": "json",
    }
    response = requests.get(f"{BASE_URL}/datasets/FUELHH/stream", params=params)
    response.raise_for_status()
    raw = response.json()

    # The FUELHH endpoint returns all fuel types — filter to WIND only
    records = [
        {"startTime": r["startTime"], "actual_mw": r["generation"]}
        for r in raw
        if r.get("fuelType") == "WIND"
    ]

    df = pd.DataFrame(records)
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
    return df.sort_values("startTime").reset_index(drop=True)


def fetch_forecasts(date_from: str, date_to: str) -> pd.DataFrame:
    """
    Fetches wind generation forecasts from the WINDFOR endpoint.
    Returns a DataFrame with columns: startTime, publishTime, forecast_mw, horizon_hrs
    Applies: Jan 2025 floor, horizon 0-48 hrs only.
    """
    params = {
        "publishDateTimeFrom": date_from,
        "publishDateTimeTo": date_to,
        "format": "json",
    }
    response = requests.get(f"{BASE_URL}/datasets/WINDFOR/stream", params=params)
    response.raise_for_status()
    raw = response.json()

    records = [
        {
            "startTime": r["startTime"],
            "publishTime": r["publishTime"],
            "forecast_mw": r["generation"],
        }
        for r in raw
    ]

    df = pd.DataFrame(records)
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
    df["publishTime"] = pd.to_datetime(df["publishTime"], utc=True)

    # horizon = startTime - publishTime, expressed in hours
    df["horizon_hrs"] = (df["startTime"] - df["publishTime"]).dt.total_seconds() / 3600

    # Apply constraints from the project spec
    jan_2025 = pd.Timestamp("2025-01-01", tz="UTC")
    df = df[(df["startTime"] >= jan_2025) & (df["horizon_hrs"].between(0, 48))]

    return df.sort_values(["startTime", "publishTime"]).reset_index(drop=True)

In [ ]:
DATE_FROM = "2025-01-01T00:00:00Z"
DATE_TO   = "2025-03-31T23:30:00Z"

df_actuals   = fetch_actuals(DATE_FROM, DATE_TO)
df_forecasts = fetch_forecasts(DATE_FROM, DATE_TO)

print(f"Actuals:   {df_actuals.shape}")
print(f"Forecasts: {df_forecasts.shape}")

df_actuals.head()

## 2. Data Cleaning & Alignment

**Horizon definition (consistent with the dashboard):**

```
horizon = startTime - publishTime
```

A forecast where `startTime = 2025-01-01T18:00Z` and `publishTime = 2025-01-01T14:00Z` has a 4-hour horizon — it was made 4 hours before the period it predicts. This matches exactly how the dashboard horizon slider works: show the latest forecast where `startTime − publishTime >= selectedHorizonHours`.

For this analysis, we compute error across all horizons rather than fixing a single value — this lets us see how accuracy degrades as the forecast is made further in advance.

**Your task:** Inspect both DataFrames, identify any data quality issues, and produce a joined DataFrame with one row per `(startTime, horizon_hrs)` containing both actual and forecast values.

Things worth checking:
- Are there duplicate `startTime` entries in actuals?
- Are there missing half-hour slots (gaps in the time series)?
- Are there zero or negative generation values that look suspicious?
- How many target times have no matching forecast? (per spec: ignore and skip these)

In [ ]:
# TODO: Inspect and clean the data.
# Document every assumption you make as a comment or markdown cell.

## 3. Forecast Error Analysis

### 3.1 Basic Error Metrics

**Your task:** Compute error for each forecast-actual pair, then summarise.

Metrics to compute: mean error, median error, MAE, RMSE, p99 absolute error.

**Think about:** Mean error alone can be misleading — a model that's always +500 MW on odd hours and -500 MW on even hours has a mean error of zero but is never accurate. That's why you need MAE/RMSE alongside it. Explain this in a markdown cell.

In [ ]:
# TODO: Compute error columns and print a summary stats table.

### 3.2 Error vs Forecast Horizon

**Your task:** Group forecasts by horizon (in hours) and plot how MAE changes as the forecast is made further in advance.

**Think about:** Does accuracy degrade linearly or are there step-changes at certain horizons? What might cause that pattern? Write your interpretation below the chart.

In [ ]:
# TODO: Group by horizon_hrs, compute MAE per bucket, plot.

### 3.3 Error by Time of Day

**Your task:** Extract hour-of-day from `startTime` and plot mean absolute error by hour.

**Think about:** Wind patterns in the UK have a diurnal cycle. Are forecast errors higher at night? In the early morning? What's your explanation for what you see?

In [ ]:
# TODO: Extract hour of day, group, plot.

## 4. Wind Generation Reliability

The question: **how many MW of wind power can we reliably expect to be available to meet electricity demand?**

This is a capacity planning question — not about averages, but about what you can depend on. The answer isn't the mean generation figure; it's a lower percentile that represents a floor you can plan around. Work through the distribution of actual generation and build your case.

### 4.1 Distribution of Actual Generation

**Your task:** Plot a histogram of actual generation values.

**Think about:** Is the distribution symmetric, skewed, bimodal? What does the shape tell you about how wind actually behaves?

In [ ]:
# TODO: Plot histogram of df_actuals['actual_mw'].

### 4.2 Percentile Analysis

**Your task:** Compute and display key percentiles (p5, p10, p25, p50, p75, p90). Mark them on the histogram.

**Think about:** p10 means wind produces at least X MW 90% of the time. p25 means at least Y MW 75% of the time. Which threshold is appropriate for grid planning depends on how much risk you're willing to accept.

In [ ]:
# TODO: Compute percentiles, annotate histogram.

### 4.3 Recommendation

**Your task:** Write your recommendation here as a markdown cell.

Structure it as:
1. What number do you recommend (X MW)?
2. What percentile does that correspond to?
3. Why is that the right threshold — what's the cost of being wrong in each direction?
4. What caveats apply — seasonality, size of the data window, extreme weather events?

There is no correct answer — the quality of your reasoning is what matters.

*Write your recommendation here.*

## 5. Summary

**Your task:** A short summary (5–8 bullet points) of the key findings from both analyses. Imagine you're presenting this to a non-technical stakeholder who needs the headline numbers.

*Write your summary here.*